# 07 — Trip Duration Prediction Agent

**Type:** Side project  
**Purpose:** Train a trip-duration model on Silver taxi data **enriched with historical weather, congestion, and holiday features**, then wire it to an LLM agent that fetches live context and returns a duration estimate.

---

### Architecture

```
User: "I'm at Times Square, taxi to JFK please"
         │
         ▼
   LLM Agent (Llama 3.3 70B)
         │
   ┌─────┴──────────────────────────────────────┐
   │  Tool calls (function calling)            │
   │  1. get_datetime_features()                │  ← current NYC time + is_holiday
   │  2. geocode_address(pickup)                │  ← lat/lon via Nominatim
   │  3. geocode_address(dropoff)               │  ← lat/lon via Nominatim
   │  4. get_weather(lat, lon)                  │  ← Open-Meteo live forecast
   │  5. get_congestion(hour, day)              │  ← historical avg demand
   │  6. predict_duration(all features)         │  ← trained HistGBT model
   └────────────────────────────────────────────┘
         │
         ▼
   "Based on current conditions (3pm Thursday, light rain,
    high congestion), your trip should take ~42 min (~12.4 mi)."
```

### Training data enrichment

| Source | Features | Join key | Inference source |
|--------|----------|----------|------------------|
| **Silver taxi data** | location, distance, time, passengers, rate code | — | User query + geocoding |
| **Open-Meteo Archive** | temperature, precipitation, snowfall, wind speed | date + hour | Open-Meteo live forecast API |
| **Congestion proxy** | hourly trip count (taxi demand = traffic proxy) | date + hour | Historical avg for that hour/day |
| **Holiday calendar** | is_holiday flag | date | Python date lookup |

### Key design decisions

| Decision | Rationale |
|----------|-----------|
| **Duration, not wait time** | Dataset only has pickup→dropoff events, not order→pickup. |
| **Duration capped at 60 min** | P99.5 = 61 min. Capping retains 99.43% of data, removes long-tail outliers. |
| **Weather joined by date+hour** | Historical weather at Central Park — city-wide, matches all trips in that hour. |
| **Congestion = taxi trip count** | No traffic API for 2015-2016; taxi volume is a strong congestion proxy. |
| **Holidays** | 5 US federal holidays in dataset range; dramatically change traffic patterns. |
| **Haversine × 1.3 for distance** | At inference we only have coordinates; road distance ≈ straight-line × 1.3 for NYC. |

## Setup

In [0]:
%pip install requests mlflow[databricks] pytz --quiet

In [0]:
%restart_python

In [0]:
import importlib
import json
import math

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import src.constants
import src.transforms
from mlflow.deployments import get_deploy_client
from pyspark.sql import functions as F
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

importlib.reload(src.constants)
importlib.reload(src.transforms)

from src.constants import (  # noqa: E402
    AGENT_LLM_ENDPOINT,
    DURATION_CAP_MIN,
    DURATION_MODEL_NAME,
    HOLIDAYS,
    LAT_LON_BIN_SIZE,
    ML_DURATION_TARGET_COLUMN,
    ML_FEATURE_COLUMNS,
    ML_RANDOM_STATE,
    ML_SAMPLE_FRACTION,
    ML_TEST_SIZE,
    SILVER_TABLE,
    WEATHER_TABLE,
)

deploy_client = get_deploy_client("databricks")
mlflow.tracing.enable()

print("Setup complete")
print(f"  Duration target : {ML_DURATION_TARGET_COLUMN}")
print(f"  Duration cap    : {DURATION_CAP_MIN} min")
print(f"  Weather table   : {WEATHER_TABLE}")
print(f"  Model name      : {DURATION_MODEL_NAME}")
print(f"  LLM endpoint    : {AGENT_LLM_ENDPOINT}")

## Enriched Feature Table

Joins Silver taxi data with:
1. **Weather** (Open-Meteo Archive, via Delta table from `07b_weather_history.ipynb`) by date + hour
2. **Congestion proxy** (city-wide hourly taxi trip count, self-derived from Silver) by date + hour
3. **Holiday flag** (US federal holidays within date range)

Duration capped at 60 min (P99.5 = 61). This removes 0.57% of rows and eliminates the extreme outliers that were destroying RMSE in the baseline model.

In [0]:
# ── Read Silver & weather ─────────────────────────────────────────────────────
silver_df = spark.read.table(SILVER_TABLE)
weather_df = spark.read.table(WEATHER_TABLE)
print(f"Silver rows:  {silver_df.count():,}")
print(f"Weather rows: {weather_df.count():,}")

# ── Add date + hour columns for joins ────────────────────────────────────────
silver_df = silver_df.withColumn(
    "trip_date", F.to_date("tpep_pickup_datetime")
).withColumn("trip_hour", F.hour("tpep_pickup_datetime"))

# ── 1) Join weather by date + hour ───────────────────────────────────────────
feature_df = silver_df.join(
    weather_df,
    (silver_df.trip_date == F.to_date(weather_df.date))
    & (silver_df.trip_hour == weather_df.hour),
    "inner",
).drop(weather_df.date, weather_df.hour)

print(f"After weather join: {feature_df.count():,}")

# ── 2) Congestion proxy: city-wide trip count per (date, hour) ───────────────
congestion_df = silver_df.groupBy("trip_date", "trip_hour").agg(
    F.count("*").alias("hourly_trip_count")
)
feature_df = feature_df.join(
    congestion_df.withColumnRenamed("trip_date", "c_date").withColumnRenamed(
        "trip_hour", "c_hour"
    ),
    (feature_df.trip_date == F.col("c_date"))
    & (feature_df.trip_hour == F.col("c_hour")),
    "left",
).drop("c_date", "c_hour")

# ── 3) Holiday flag ──────────────────────────────────────────────────────────
holiday_list = list(HOLIDAYS)
feature_df = feature_df.withColumn(
    "is_holiday",
    F.when(F.col("trip_date").cast("string").isin(holiday_list), 1).otherwise(0),
)

# ── 4) Cap duration at 60 min ────────────────────────────────────────────────
feature_df = feature_df.filter(
    (F.col(ML_DURATION_TARGET_COLUMN) > 0)
    & (F.col(ML_DURATION_TARGET_COLUMN) <= DURATION_CAP_MIN)
)

# ── Select features + target ─────────────────────────────────────────────────
enriched_features = [
    *ML_FEATURE_COLUMNS,
    "temperature_f",
    "precipitation_inch",
    "snowfall_inch",
    "wind_speed_mph",
    "hourly_trip_count",
    "is_holiday",
]
feature_df = feature_df.select(*enriched_features, ML_DURATION_TARGET_COLUMN)

# Drop nulls (rate_code_id NULLs, empty zones, weather nulls)
feature_df = feature_df.dropna()
feature_df = feature_df.filter(
    (F.col("pickup_zone") != "") & (F.col("dropoff_zone") != "")
)

# ── Parse zone strings into numeric lat/lon bins ─────────────────────────────
for prefix in ["pickup", "dropoff"]:
    feature_df = feature_df.withColumn(
        f"{prefix}_lat_bin",
        F.split(F.col(f"{prefix}_zone"), ",")[0].cast("double"),
    ).withColumn(
        f"{prefix}_lon_bin",
        F.split(F.col(f"{prefix}_zone"), ",")[1].cast("double"),
    )
feature_df = feature_df.drop("pickup_zone", "dropoff_zone")

# Cast booleans → int for sklearn
feature_df = feature_df.withColumn("is_weekend", F.col("is_weekend").cast("int"))

# ── Sample ────────────────────────────────────────────────────────────────────
feature_df = feature_df.sample(fraction=ML_SAMPLE_FRACTION, seed=ML_RANDOM_STATE)
pdf = feature_df.toPandas()

# One-hot encode rate_code_id
pdf = pd.get_dummies(pdf, columns=["rate_code_id"], prefix="rc", dtype=int)

print(f"\nEnriched feature table: {pdf.shape}")
print(
    f"Target mean: {pdf[ML_DURATION_TARGET_COLUMN].mean():.1f} min "
    f"| std: {pdf[ML_DURATION_TARGET_COLUMN].std():.1f} min "
    f"| max: {pdf[ML_DURATION_TARGET_COLUMN].max():.0f} min"
)

X = pdf.drop(columns=[ML_DURATION_TARGET_COLUMN])
y = pdf[ML_DURATION_TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=ML_TEST_SIZE, random_state=ML_RANDOM_STATE
)
print(f"Train: {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
print(f"Test:  {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(
    "\nNew features: temperature_f, precipitation_inch, snowfall_inch, "
    "wind_speed_mph, hourly_trip_count, is_holiday"
)

# ── Build congestion lookup for inference ─────────────────────────────────────
# Average hourly trip count by (hour_of_day, day_of_week) across all dates
congestion_lookup = (
    congestion_df.withColumn("dow", F.dayofweek("trip_date"))
    .groupBy(F.col("trip_hour").alias("lk_hour"), F.col("dow").alias("lk_dow"))
    .agg(F.round(F.avg("hourly_trip_count")).cast("int").alias("avg_trips"))
    .toPandas()
)
congestion_lookup = dict(
    zip(
        zip(congestion_lookup["lk_hour"], congestion_lookup["lk_dow"]),
        congestion_lookup["avg_trips"],
    )
)
congestion_median = int(np.median(list(congestion_lookup.values())))
print(f"\nCongestion lookup: {len(congestion_lookup)} (hour, day_of_week) entries")
print(f"Median hourly trips: {congestion_median:,}")

## Duration Model — Train & Register

In [0]:
# ── Train HistGradientBoostingRegressor on enriched features ──────────────────
dur_params = {
    "max_iter": 300,
    "max_depth": 8,
    "learning_rate": 0.05,
    "min_samples_leaf": 50,
    "random_state": ML_RANDOM_STATE,
}

duration_model = HistGradientBoostingRegressor(**dur_params)
duration_model.fit(X_train, y_train)

# ── Evaluate ──────────────────────────────────────────────────────────────────
y_pred = duration_model.predict(X_test)
dur_metrics = {
    "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
    "mae": float(mean_absolute_error(y_test, y_pred)),
    "r2": float(r2_score(y_test, y_pred)),
}

# ── Log & register to MLflow ──────────────────────────────────────────────────
input_example = X_train.iloc[:5]

with mlflow.start_run(run_name="duration_enriched_HistGBT"):
    mlflow.log_params(dur_params)
    mlflow.log_param("model_type", "HistGradientBoostingRegressor")
    mlflow.log_param("target", ML_DURATION_TARGET_COLUMN)
    mlflow.log_param("duration_cap_min", DURATION_CAP_MIN)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_rows", X_train.shape[0])
    mlflow.log_param("test_rows", X_test.shape[0])
    mlflow.log_param("enrichment", "weather+congestion+holidays")
    mlflow.log_metrics(dur_metrics)
    mlflow.sklearn.log_model(
        duration_model,
        name="model",
        input_example=input_example,
        registered_model_name=DURATION_MODEL_NAME,
    )
    dur_run_id = mlflow.active_run().info.run_id

# Store feature column order for inference alignment
duration_feature_cols = list(X_train.columns)

avg_duration_overall = float(y_train.mean())
avg_duration_by_hour = y_train.groupby(X_train["hour_of_day"]).mean().to_dict()
print(f"  Avg duration (all): {avg_duration_overall:.1f} min")

print("Trip Duration Model — Enriched HistGradientBoosting")
print("=" * 50)
print(f"  RMSE:  {dur_metrics['rmse']:.2f} min")
print(f"  MAE:   {dur_metrics['mae']:.2f} min")
print(f"  R²:    {dur_metrics['r2']:.4f}")
print(f"\n  Features: {X_train.shape[1]} (base + weather + congestion + holidays)")
print(f"  Duration cap: ≤{DURATION_CAP_MIN} min")
print(f"  MLflow run ID: {dur_run_id}")
print(f"  Registered as: {DURATION_MODEL_NAME}")

## Agent Tools

Five tools that the LLM can call during a conversation. Every feature passed to the model at inference was also present during training.

| Tool | Data source | Features for model | Notes |
|------|-------------|-------------------|-------|
| `get_datetime_features` | System clock | hour_of_day, day_of_week, is_weekend, is_holiday | Current NYC time |
| `geocode_address` | Nominatim (OSM) | pickup/dropoff lat/lon → zone bins + haversine distance | Free, no API key |
| `get_weather` | Open-Meteo forecast | temperature_f, precipitation_inch, snowfall_inch, wind_speed_mph | Same features as training (historical archive) |
| `get_congestion` | Historical avg from training data | hourly_trip_count | Lookup by (hour, day_of_week) |
| `predict_duration` | Trained HistGBT model | All above combined | Returns predicted minutes + road distance |

In [0]:
import requests
import pytz
from datetime import datetime

# Spark dayofweek convention: 1=Sun, 2=Mon, ..., 7=Sat
# Python weekday():           0=Mon, 1=Tue, ..., 6=Sun
_PYTHON_TO_SPARK_DOW = {0: 2, 1: 3, 2: 4, 3: 5, 4: 6, 5: 7, 6: 1}


def get_datetime_features() -> dict:
    """Return current NYC time as model features: hour, day, weekend, holiday."""
    now = datetime.now(pytz.timezone("America/New_York"))
    python_dow = now.weekday()
    date_str = now.strftime("%Y-%m-%d")
    return {
        "hour_of_day": now.hour,
        "day_of_week": _PYTHON_TO_SPARK_DOW[python_dow],
        "is_weekend": 1 if python_dow >= 5 else 0,
        "is_holiday": 1 if date_str in HOLIDAYS else 0,
        "day_name": now.strftime("%A"),
        "current_time": now.strftime("%Y-%m-%d %H:%M %Z"),
    }


def geocode_address(address: str) -> dict:
    """Geocode an NYC address or landmark to lat/lon using Nominatim (OpenStreetMap)."""
    resp = requests.get(
        "https://nominatim.openstreetmap.org/search",
        params={"q": f"{address}, New York City", "format": "json", "limit": 1},
        headers={"User-Agent": "nyc-taxi-duration-agent/1.0"},
        timeout=10,
    )
    resp.raise_for_status()
    results = resp.json()
    if not results:
        raise ValueError(f"Could not geocode address: {address!r}")
    return {
        "lat": float(results[0]["lat"]),
        "lon": float(results[0]["lon"]),
        "display_name": results[0]["display_name"],
    }


def get_weather(lat: float, lon: float) -> dict:
    """Get current weather — same features the model was trained on (Open-Meteo Archive)."""
    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat,
            "longitude": lon,
            "current": "temperature_2m,precipitation,snowfall,wind_speed_10m",
            "temperature_unit": "fahrenheit",
            "wind_speed_unit": "mph",
            "precipitation_unit": "inch",
            "timezone": "America/New_York",
        },
        timeout=10,
    )
    resp.raise_for_status()
    c = resp.json()["current"]
    return {
        "temperature_f": c["temperature_2m"],
        "precipitation_inch": c["precipitation"],
        "snowfall_inch": c["snowfall"],
        "wind_speed_mph": c["wind_speed_10m"],
    }


def get_congestion(hour_of_day: int, day_of_week: int) -> dict:
    """Look up historical average taxi demand for this hour and day of week.

    The model was trained on actual hourly trip counts; at inference we use the
    historical average as a proxy for current congestion.
    """
    key = (hour_of_day, day_of_week)
    avg_trips = congestion_lookup.get(key, congestion_median)
    return {"hourly_trip_count": int(avg_trips)}


def predict_duration(
    pickup_lat: float,
    pickup_lon: float,
    dropoff_lat: float,
    dropoff_lon: float,
    hour_of_day: int,
    day_of_week: int,
    is_weekend: int,
    is_holiday: int,
    temperature_f: float,
    precipitation_inch: float,
    snowfall_inch: float,
    wind_speed_mph: float,
    hourly_trip_count: int,
    passenger_count: int = 1,
    rate_code_id: int = 1,
) -> dict:
    """Predict NYC taxi trip duration using the enriched HistGBT model.

    All features (weather, congestion, holidays) were present during training.
    trip_distance is approximated as haversine × 1.3.
    """
    bin_size = LAT_LON_BIN_SIZE

    pickup_lat_bin = round(pickup_lat / bin_size) * bin_size
    pickup_lon_bin = round(pickup_lon / bin_size) * bin_size
    dropoff_lat_bin = round(dropoff_lat / bin_size) * bin_size
    dropoff_lon_bin = round(dropoff_lon / bin_size) * bin_size

    # Haversine distance (miles) × 1.3 road-distance correction
    R = 3958.8
    lat1, lon1 = math.radians(pickup_lat), math.radians(pickup_lon)
    lat2, lon2 = math.radians(dropoff_lat), math.radians(dropoff_lon)
    a = (
        math.sin((lat2 - lat1) / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin((lon2 - lon1) / 2) ** 2
    )
    trip_distance = round(2 * R * math.asin(math.sqrt(a)) * 1.3, 2)

    # One-hot encode rate_code_id
    rc_cols = {f"rc_{i}": (1 if rate_code_id == i else 0) for i in range(1, 7)}

    row = {
        "hour_of_day": hour_of_day,
        "day_of_week": day_of_week,
        "is_weekend": is_weekend,
        "trip_distance": trip_distance,
        "passenger_count": passenger_count,
        "temperature_f": temperature_f,
        "precipitation_inch": precipitation_inch,
        "snowfall_inch": snowfall_inch,
        "wind_speed_mph": wind_speed_mph,
        "hourly_trip_count": hourly_trip_count,
        "is_holiday": is_holiday,
        "pickup_lat_bin": pickup_lat_bin,
        "pickup_lon_bin": pickup_lon_bin,
        "dropoff_lat_bin": dropoff_lat_bin,
        "dropoff_lon_bin": dropoff_lon_bin,
        **rc_cols,
    }
    features = pd.DataFrame([row])

    # Align to training column order; fill any unseen OHE columns with 0
    for col in duration_feature_cols:
        if col not in features.columns:
            features[col] = 0
    features = features[duration_feature_cols]

    predicted_min = max(1.0, float(duration_model.predict(features)[0]))
    pct_vs_avg = round((predicted_min - avg_duration_overall) / avg_duration_overall * 100, 1)
    hour_avg = avg_duration_by_hour.get(hour_of_day, avg_duration_overall)
    pct_vs_hour = round((predicted_min - hour_avg) / hour_avg * 100, 1)
    return {
        "predicted_duration_min": round(predicted_min, 1),
        "estimated_road_distance_miles": trip_distance,
        "avg_duration_all_trips_min": round(avg_duration_overall, 1),
        "pct_vs_overall_avg": pct_vs_avg,
        "avg_duration_this_hour_min": round(hour_avg, 1),
        "pct_vs_hour_avg": pct_vs_hour,
    }


print(
    "Tools defined: get_datetime_features, geocode_address, get_weather, "
    "get_congestion, predict_duration"
)

## Trip Duration Agent

The agent uses Databricks Foundation Model API (OpenAI-compatible) with tool/function calling.  
The LLM orchestrates all tool calls autonomously — the user just asks in plain English.

In [0]:
_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_datetime_features",
            "description": "Get the current NYC date and time as model features (hour_of_day, day_of_week, is_weekend, is_holiday). Always call this first.",
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "geocode_address",
            "description": "Convert an NYC street address or landmark name to latitude/longitude coordinates.",
            "parameters": {
                "type": "object",
                "properties": {
                    "address": {
                        "type": "string",
                        "description": "Street address or NYC landmark (e.g. 'Times Square', '34th St Penn Station', 'JFK Airport')",
                    }
                },
                "required": ["address"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather conditions (temperature, precipitation, snowfall, wind). These are features the ML model was trained on.",
            "parameters": {
                "type": "object",
                "properties": {
                    "lat": {"type": "number", "description": "Latitude"},
                    "lon": {"type": "number", "description": "Longitude"},
                },
                "required": ["lat", "lon"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_congestion",
            "description": "Get the historical average taxi demand (trip count) for this hour and day of week. This is a congestion proxy the ML model was trained on.",
            "parameters": {
                "type": "object",
                "properties": {
                    "hour_of_day": {"type": "integer", "description": "Hour (0-23)"},
                    "day_of_week": {
                        "type": "integer",
                        "description": "Spark convention: 1=Sun, 2=Mon, ..., 7=Sat",
                    },
                },
                "required": ["hour_of_day", "day_of_week"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "predict_duration",
            "description": "Predict NYC taxi trip duration using the trained ML model. You MUST pass ALL features: location coords, time, weather, congestion, and holiday flag. Call this last after collecting all inputs from the other tools.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pickup_lat": {"type": "number", "description": "Pickup latitude"},
                    "pickup_lon": {"type": "number", "description": "Pickup longitude"},
                    "dropoff_lat": {
                        "type": "number",
                        "description": "Dropoff latitude",
                    },
                    "dropoff_lon": {
                        "type": "number",
                        "description": "Dropoff longitude",
                    },
                    "hour_of_day": {
                        "type": "integer",
                        "description": "Current hour (0-23)",
                    },
                    "day_of_week": {
                        "type": "integer",
                        "description": "1=Sun, 2=Mon, ..., 7=Sat",
                    },
                    "is_weekend": {
                        "type": "integer",
                        "description": "1 if Sat/Sun, else 0",
                    },
                    "is_holiday": {
                        "type": "integer",
                        "description": "1 if US federal holiday, else 0",
                    },
                    "temperature_f": {
                        "type": "number",
                        "description": "Current temperature in Fahrenheit",
                    },
                    "precipitation_inch": {
                        "type": "number",
                        "description": "Current precipitation in inches",
                    },
                    "snowfall_inch": {
                        "type": "number",
                        "description": "Current snowfall in inches",
                    },
                    "wind_speed_mph": {
                        "type": "number",
                        "description": "Current wind speed in mph",
                    },
                    "hourly_trip_count": {
                        "type": "integer",
                        "description": "Historical avg taxi trips for this hour/day",
                    },
                    "passenger_count": {
                        "type": "integer",
                        "description": "Number of passengers (default 1)",
                    },
                    "rate_code_id": {
                        "type": "integer",
                        "description": "TLC rate code: 1=Standard, 2=JFK, 3=Newark (default 1)",
                    },
                },
                "required": [
                    "pickup_lat",
                    "pickup_lon",
                    "dropoff_lat",
                    "dropoff_lon",
                    "hour_of_day",
                    "day_of_week",
                    "is_weekend",
                    "is_holiday",
                    "temperature_f",
                    "precipitation_inch",
                    "snowfall_inch",
                    "wind_speed_mph",
                    "hourly_trip_count",
                ],
            },
        },
    },
]

_TOOL_DISPATCH = {
    "get_datetime_features": get_datetime_features,
    "geocode_address": geocode_address,
    "get_weather": get_weather,
    "get_congestion": get_congestion,
    "predict_duration": predict_duration,
}

_SYSTEM_PROMPT = """You are a helpful NYC taxi trip duration assistant.

When a user asks how long a taxi journey will take, you MUST follow these steps IN ORDER:
1. Call get_datetime_features() to get the current time, day, and holiday status.
2. Call geocode_address() for the pickup location.
3. Call geocode_address() for the dropoff location.
4. Call get_weather() at the pickup coordinates.
5. Call get_congestion() with the hour_of_day and day_of_week from step 1.
6. Call predict_duration() passing ALL collected data: coordinates, time features,
   weather features, congestion, and holiday flag.
7. Respond with the estimated duration, road distance, and a brief note on conditions
   (weather, congestion level, time of day, holiday status).

IMPORTANT:
- The ML model was trained on weather, congestion, and holiday data alongside taxi trip data.
  Every feature must be passed to predict_duration — do NOT skip any.
- Always use the tools — never guess coordinates, distances, or durations.
- If a location is ambiguous, ask the user to clarify before calling tools.

The format of your response should be 1. A VERY short sentence saying the predicted time and if it is longer or shorter than average (with a value showing how much lower), and 2. In another paragraph, a very brief description of the other conditions returned by the other tools and why it came to that conclusion"""


def ask_agent(user_input: str) -> str:
    """Run the trip duration agent with a natural-language query."""
    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user", "content": user_input},
    ]

    for _ in range(10):  # max 10 tool-call rounds
        response = deploy_client.predict(
            endpoint=AGENT_LLM_ENDPOINT,
            inputs={
                "messages": messages,
                "tools": _TOOLS_SCHEMA,
                "tool_choice": "auto",
            },
        )
        msg = response.choices[0]["message"]

        if not msg.get("tool_calls"):
            return msg["content"]

        messages.append(msg)
        for tc in msg["tool_calls"]:
            fn_name = tc["function"]["name"]
            fn_args = json.loads(tc["function"]["arguments"])
            print(
                f"  → {fn_name}({', '.join(f'{k}={v!r}' for k, v in fn_args.items())})"
            )
            result = _TOOL_DISPATCH[fn_name](**fn_args)
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "content": json.dumps(result),
                }
            )

    return "Agent did not reach a conclusion within the tool-call limit."


print("Agent ready.")
print(
    "Call: ask_agent('I am at Times Square and want a taxi to JFK. How long will it take?')"
)

## Demo

In [0]:
query = "I'm at Times Square and want a taxi to Penn Station. How long will it take?"
print(f"User: {query}\n")
print("Tool calls:")
answer = ask_agent(query)
print(f"\nAgent: {answer}")

In [0]:
# Try another query
query2 = (
    "How long would a yellow cab take from JFK airport to the Empire State Building?"
)
print(f"User: {query2}\n")
print("Tool calls:")
answer2 = ask_agent(query2)
print(f"\nAgent: {answer2}")